# 🔬 Notebook 3: Google Calendar — Deep Dives

## 🛠️ Setup

```bash
cd 06-system-designs/google-calendar
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


This notebook zooms into three of the trickiest algorithms in a calendar service:

1. **Recurrence expansion** — turning one RRULE into the list of concrete occurrences inside a query window.
2. **Exception handling** — moving or cancelling a *single instance* of a recurring event.
3. **Free/busy (availability)** — answering "when are all of these people free for 30 minutes?"

We show each as **bad practice → best practice** so the *why* is obvious.

## 1. Recurrence expansion

### ❌ Bad: materialize every future occurrence at write time

Imagine the user creates "Daily stand-up at 9am" with no end date. The "simple" approach is to insert rows for every day for, say, the next 5 years.

- 5 years × 365 days = 1,825 rows **per event**.
- Changing the time means updating 1,825 rows.
- Storage explodes; updates become transactional nightmares.

### ✅ Best: store the **rule**, expand on read

Store one row with `rrule='FREQ=DAILY;BYHOUR=9'`. When a client asks for a specific week, expand just those ~7 rows in memory.

Below is a toy DAILY expander (easy to follow) followed by the **real** way using `python-dateutil`, which supports full iCalendar RRULE semantics (BYDAY, BYMONTHDAY, UNTIL, COUNT, EXDATE, ...).

In [1]:
# Toy DAILY expander -- good for intuition, bad for production
from datetime import datetime, timedelta, timezone

def expand_daily(start, interval_days, window_from, window_to):
    out, t = [], start
    while t < window_from:
        t += timedelta(days=interval_days)
    while t <= window_to:
        out.append(t)
        t += timedelta(days=interval_days)
    return out

start = datetime(2026, 1, 1, 9, tzinfo=timezone.utc)
for x in expand_daily(
    start, 2,
    datetime(2026, 1, 5, tzinfo=timezone.utc),
    datetime(2026, 1, 15, tzinfo=timezone.utc),
):
    print(x)


2026-01-05 09:00:00+00:00
2026-01-07 09:00:00+00:00
2026-01-09 09:00:00+00:00
2026-01-11 09:00:00+00:00
2026-01-13 09:00:00+00:00


In [2]:
# Production-grade: python-dateutil understands the iCalendar RFC 5545 RRULE
from dateutil.rrule import rrulestr
from datetime import datetime, timezone

# Every weekday at 9:00 UTC, starting Jan 5 2026, 10 occurrences
rule = rrulestr(
    "FREQ=WEEKLY;BYDAY=MO,TU,WE,TH,FR;COUNT=10",
    dtstart=datetime(2026, 1, 5, 9, tzinfo=timezone.utc),
)

# Ask only for a 1-week window
for dt in rule.between(
    datetime(2026, 1, 5, tzinfo=timezone.utc),
    datetime(2026, 1, 9, 23, 59, tzinfo=timezone.utc),
    inc=True,
):
    print(dt)


2026-01-05 09:00:00+00:00
2026-01-06 09:00:00+00:00
2026-01-07 09:00:00+00:00
2026-01-08 09:00:00+00:00
2026-01-09 09:00:00+00:00


## 2. Overriding or cancelling one occurrence

Users often want to tweak **just one instance** of a recurring event:

- "Move *next* Monday's 1:1 to Tuesday" (override).
- "Skip the 1:1 this week — I'm on vacation" (cancel).

### Data model

```
events          : (id, rrule, starts_at, ends_at, ...)
event_exceptions: (event_id, original_start, new_start NULLABLE, cancelled BOOL)
```

We keep the rule intact and store a small delta keyed by the occurrence's **original** start time. On read, we merge: for every expanded occurrence, look up an exception; if it's cancelled, drop it; if it has a `new_start`, replace it.

In [3]:
from dateutil.rrule import rrulestr
from datetime import datetime, timedelta, timezone

rule = rrulestr(
    "FREQ=WEEKLY;BYDAY=MO;COUNT=8",
    dtstart=datetime(2026, 5, 4, 15, tzinfo=timezone.utc),
)

# Exception table (in-memory for the demo)
#   key   = original occurrence start (UTC)
#   value = None -> cancelled
#   value = dt   -> moved to this new start
exceptions = {}

def move(original_start, new_start):
    exceptions[original_start] = new_start

def cancel(original_start):
    exceptions[original_start] = None

def occurrences_in(window_from, window_to):
    result = []
    for original in rule.between(window_from, window_to, inc=True):
        if original in exceptions:
            ex = exceptions[original]
            if ex is None:
                continue
            result.append((original, ex))
        else:
            result.append((original, original))
    return result

move(datetime(2026, 5, 11, 15, tzinfo=timezone.utc),
     datetime(2026, 5, 12, 15, tzinfo=timezone.utc))
cancel(datetime(2026, 5, 18, 15, tzinfo=timezone.utc))

for original, shown in occurrences_in(
    datetime(2026, 5,  1, tzinfo=timezone.utc),
    datetime(2026, 6,  1, tzinfo=timezone.utc),
):
    tag = "(moved)" if original != shown else ""
    print(shown, tag)


2026-05-04 15:00:00+00:00 
2026-05-12 15:00:00+00:00 (moved)
2026-05-25 15:00:00+00:00 


**Why key on `original_start` and not on a synthetic `occurrence_id`?** Because when the base event's time shifts, the rule changes and synthetic IDs drift. The original start time is a stable identity for "this particular instance of the series".

## 3. Free/busy — the heart of availability

Question: *"Find me 30 minutes between 9am–5pm today when Alice, Bob, and Room 7 are **all** free."*

This is how "Find a time" and the room-booking suggester work.

### Mental model

Each attendee's calendar for the day is a set of **busy intervals**. An attendee is free precisely when they are NOT inside any busy interval. The whole meeting works only if *every* attendee is free at the same time.

So we need to:

1. Pull each attendee's busy intervals in the window.
2. **Merge** them into a single set of "someone is busy" intervals.
3. Walk through the gaps and return any gap ≥ the requested duration.

### ❌ Bad: minute-by-minute scan

Beginners often write this: split the window into 1-minute slots, mark each slot busy if *any* attendee is busy, then scan for runs of free minutes.

- For a 9-hour window that's 540 slots. With N attendees and M events per attendee it's **O(slots · N · M)**.
- Works for a demo, dies at scale (imagine scanning a year for 50 people).

In [4]:
from datetime import datetime, timedelta, timezone

def find_slot_naive(busy_per_person, window_from, window_to, duration_min):
    total_minutes = int((window_to - window_from).total_seconds() // 60)
    busy_mask = [False] * total_minutes
    for busy_list in busy_per_person:
        for (s, e) in busy_list:
            i = max(0, int((s - window_from).total_seconds() // 60))
            j = min(total_minutes, int((e - window_from).total_seconds() // 60))
            for k in range(i, j):
                busy_mask[k] = True

    run = 0
    for idx, b in enumerate(busy_mask):
        run = 0 if b else run + 1
        if run >= duration_min:
            start_idx = idx - duration_min + 1
            return window_from + timedelta(minutes=start_idx)
    return None

W0 = datetime(2026, 5, 4,  9, tzinfo=timezone.utc)
W1 = datetime(2026, 5, 4, 17, tzinfo=timezone.utc)

alice = [(datetime(2026,5,4, 9,30,tzinfo=timezone.utc), datetime(2026,5,4,10,30,tzinfo=timezone.utc)),
         (datetime(2026,5,4,13, 0,tzinfo=timezone.utc), datetime(2026,5,4,14, 0,tzinfo=timezone.utc))]
bob   = [(datetime(2026,5,4,10, 0,tzinfo=timezone.utc), datetime(2026,5,4,11, 0,tzinfo=timezone.utc))]
room7 = [(datetime(2026,5,4,15, 0,tzinfo=timezone.utc), datetime(2026,5,4,16, 0,tzinfo=timezone.utc))]

print("First 30-min slot (naive):",
      find_slot_naive([alice, bob, room7], W0, W1, 30))


First 30-min slot (naive): 2026-05-04 09:00:00+00:00


### ✅ Best: sweep-line / interval merge — O((N·M) log (N·M))

Classic trick: take all the busy intervals together, sort their endpoints, sweep left-to-right counting how many are "open". Whenever the count is zero, everyone is free — look at the gap between the last close and the next open.

This is the same algorithm as "merge overlapping intervals" from Leetcode, just with a min-duration filter at the end.

In [5]:
from datetime import datetime, timedelta, timezone

def find_slot_sweep(busy_per_person, window_from, window_to, duration_min):
    events = []
    for busy_list in busy_per_person:
        for (s, e) in busy_list:
            s = max(s, window_from)
            e = min(e, window_to)
            if s < e:
                events.append((s, +1))
                events.append((e, -1))
    events.sort()

    duration = timedelta(minutes=duration_min)
    open_count = 0
    cursor = window_from
    for t, delta in events:
        if open_count == 0 and t - cursor >= duration:
            return cursor
        open_count += delta
        if open_count == 0:
            cursor = t
    if open_count == 0 and window_to - cursor >= duration:
        return cursor
    return None

W0 = datetime(2026, 5, 4,  9, tzinfo=timezone.utc)
W1 = datetime(2026, 5, 4, 17, tzinfo=timezone.utc)
alice = [(datetime(2026,5,4, 9,30,tzinfo=timezone.utc), datetime(2026,5,4,10,30,tzinfo=timezone.utc)),
         (datetime(2026,5,4,13, 0,tzinfo=timezone.utc), datetime(2026,5,4,14, 0,tzinfo=timezone.utc))]
bob   = [(datetime(2026,5,4,10, 0,tzinfo=timezone.utc), datetime(2026,5,4,11, 0,tzinfo=timezone.utc))]
room7 = [(datetime(2026,5,4,15, 0,tzinfo=timezone.utc), datetime(2026,5,4,16, 0,tzinfo=timezone.utc))]

print("First 30-min slot (sweep):",
      find_slot_sweep([alice, bob, room7], W0, W1, 30))


First 30-min slot (sweep): 2026-05-04 09:00:00+00:00


### Sanity check: both methods agree

A good habit when you replace a slow-but-obvious algorithm with a fast one is to **randomly compare** them on many inputs. If they ever disagree, you have a bug in the fast one.

In [6]:
import random
from datetime import datetime, timedelta, timezone

random.seed(42)
W0 = datetime(2026, 5, 4, 9, tzinfo=timezone.utc)

def rand_busy():
    out = []
    for _ in range(random.randint(0, 3)):
        start_min = random.randint(0, 7*60)
        length    = random.randint(30, 120)
        s = W0 + timedelta(minutes=start_min)
        e = s + timedelta(minutes=length)
        out.append((s, e))
    return out

mismatches = 0
for _ in range(200):
    people = [rand_busy() for _ in range(random.randint(1, 4))]
    W1 = W0 + timedelta(hours=8)
    dur = random.choice([15, 30, 45, 60])
    a = find_slot_naive(people, W0, W1, dur)
    b = find_slot_sweep(people, W0, W1, dur)
    if a != b:
        mismatches += 1

print(f"mismatches across 200 random inputs: {mismatches}")
assert mismatches == 0, "sweep disagrees with naive -- bug!"


mismatches across 200 random inputs: 0


## Real-world caveats we glossed over

Good to know these exist even if we won't implement them here:

- **All-day events** (`DATE` not `DATE-TIME`) — spanning a whole day across timezones needs care.
- **EXDATE / RDATE** — RFC 5545 lets you exclude or inject individual dates into a series; `dateutil` supports both.
- **"This and future"** edits typically *split* the series: end the old rule with an `UNTIL`, create a new event starting at the modified occurrence.
- **Working hours & DND** — free/busy should subtract time outside the user's working hours and within Do-Not-Disturb windows.
- **Resource conflicts** — treating a room as "an attendee with a calendar" is the cleanest model; booking just adds an invitation for the room.
- **Delegation & ACLs** — Alice's assistant can act on her calendar; the permission check is on every read and write.
- **Reminder delivery at scale** — don't `SELECT ... WHERE fires_at <= now()` every second. Use a partitioned durable timer queue (see `reminder-alert` lab).


## Closing thoughts

- **Store rules, expand on read** — the core trick for recurring events.
- **Model exceptions as a small delta table** keyed by `original_start`.
- **Free/busy is interval-merge** — sweep-line beats minute-scanning by orders of magnitude.
- **UTC + IANA tz** — never forget which one is the source of truth.

If you internalize those four ideas, you've got the backbone of a calendar service.
